# Google Colab file.

# Aerial object detection: planes, ships and large vehicles
Personal project. Trains a small YOLO model on public DOTA aerial photos to find planes, ships and large vehicles, then checks how well it works and converts detections to map positions.
Runs on Google Colab. Code and results are in the Drive folder aerial-vehicle-detection.

### Setup: run at the start of every session
Connects Google Drive, moves into the project folder and installs the YOLO library.

#### run steps 1, 3 and 4 again after reconnecting, once you have already done step 6 for training

In [1]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd "/content/drive/MyDrive/Colab Notebooks/aerial-vehicle-detection"
!pip install -q ultralytics
!nvidia-smi

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/aerial-vehicle-detection
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!ls

01_download_data.py	    09_predict_pictures.py	README.md
02_look_at_data.py	    10_pixel_to_latlon.py	requirements.txt
03_make_tiles.py	    11_app.py			results
04_write_yaml_and_check.py  aerial-detection-run.ipynb	runs
05_augmentation_preview.py  config.py			tools.py
06_train.py		    Docs			yolo11n.pt
07_evaluate.py		    model
08_recall_by_size.py	    __pycache__


### Step 1: Download the DOTA aerial photos
Downloads about 2 GB of photos with hand-drawn labels and unzips them to /content/data.

In [3]:
!python 01_download_data.py

Unzipping...
train images: 1411
val images: 458
test images: 937
Data is in /content/data/DOTAv1


### Step 2: Look at the data
Counts objects per class, measures their size in pixels and saves two pictures.

In [ ]:
!python 02_look_at_data.py

### Step 3: Cut the photos into tiles
Cuts the large photos into overlapping 640 x 640 tiles and writes matching label files.

In [4]:
!python 03_make_tiles.py

  train: 100 of 1411 images done
  train: 150 of 1411 images done
  train: 250 of 1411 images done
  train: 300 of 1411 images done
  train: 450 of 1411 images done
  train: 500 of 1411 images done
  train: 600 of 1411 images done
  train: 700 of 1411 images done
  train: 750 of 1411 images done
  train: 900 of 1411 images done
  train: 950 of 1411 images done
  train: 1000 of 1411 images done
  train: 1100 of 1411 images done
  train: 1150 of 1411 images done
  train: 1200 of 1411 images done
  train: 1350 of 1411 images done
  train: 1400 of 1411 images done
train: 10164 tiles, 99030 objects
  val: 50 of 458 images done
  val: 100 of 458 images done
  val: 150 of 458 images done
  val: 350 of 458 images done
  val: 400 of 458 images done
val: 3043 tiles, 30297 objects
Tiles are in /content/data/tiles


### Step 4: Write data.yaml and check the tiles
Tells YOLO where the data is, checks that no photo is in both train and validation, and draws labels on sample tiles.

In [5]:
!python 04_write_yaml_and_check.py

Wrote /content/data/tiles/data.yaml
train: 10164 tiles, objects per class: {'plane': 13374, 'ship': 50376, 'large vehicle': 35280}
val: 3043 tiles, objects per class: {'plane': 4095, 'ship': 17052, 'large vehicle': 9150}
Photos in both train and val: 0
Saved results/04_tile_check.jpg  (open it and check that the boxes sit on the objects)


### Step 5: Augmentation preview (optional)
Shows one tile flipped, turned, brighter and darker, with the boxes following.

In [ ]:
!python 05_augmentation_preview.py

### Step 6: Train the model
Fine-tunes YOLO11n on the tiles. The best model is saved in runs/optical_v1/weights/best.pt.

In [ ]:
!python 06_train.py

### Step 7: Evaluate the model
Prints precision, recall and mAP50 per class, and the speed in milliseconds per tile.

In [6]:
!python 07_evaluate.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,537 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2218.5±1019.1 MB/s, size: 126.7 KB)
val: Scanning /content/data/tiles/labels/val... 3043 images, 551 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3043/3043 1.9Kit/s 1.6s
val: New cache created: /content/data/tiles/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 191/191 6.1it/s 31.3s
                   all       3043      30297      0.914      0.887      0.937      0.699
                 plane        932       4095

### Step 8: Recall by object size
Checks how often objects of each size are found, to see if small objects are missed.

In [7]:
!python 08_recall_by_size.py

        class    object size  objects  recall
        plane    under 16 px        5    0.00
        plane       16-32 px       43    0.42
        plane       32-64 px      128    0.98
        plane 64 px and more      242    0.98
         ship    under 16 px        4    1.00
         ship       16-32 px      291    0.91
         ship       32-64 px     2053    0.97
         ship 64 px and more      297    0.96
large vehicle    under 16 px        2    0.00
large vehicle       16-32 px       35    0.17
large vehicle       32-64 px      485    0.87
large vehicle 64 px and more      556    0.91
Saved results/08_recall_by_size.png


### Step 9: Predictions on tiles and one full photo
Draws the model's detections on sample tiles and on one full-size photo.

In [9]:
!python 09_predict_pictures.py

Saved tile predictions in /content/drive/MyDrive/Colab Notebooks/aerial-vehicle-detection/results/09_predictions
Big photo: P0179.jpg with 86 real objects of my classes
The model found 123 objects
Saved results/09_big_photo.jpg


### Step 10: Pixel position to latitude and longitude
Converts each detection's position to map coordinates (made-up georeference) and saves a table.

In [6]:
!python 10_pixel_to_latlon.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
1000 px to the east : 500.0 m (should be 500)
1000 px to the south: 500.0 m (should be 500)
class  confidence  size_m       lat      lon
plane        0.89    58.1 -0.015412 0.020493
plane        0.87    45.6 -0.005903 0.008710
plane        0.87    50.1 -0.008009 0.004084
plane        0.87    39.3 -0.018434 0.018639
plane        0.87    64.1 -0.014177 0.018086
plane        0.87    73.3 -0.003315 0.001041
plane        0.87    43.6 -0.014010 0.016506
plane        0.86    74.4 -0.011386 0.012930
plane        0.86    50.7 -0.011669 0.015802
plane        0.86    45.8 -0.008796 0.006178
Saved 123 detections in results/10_detections_with_coordinates.csv


### Step 11: Web app (runs on the Mac, not here)
11_app.py is run with Streamlit on your computer.

In [5]:
# Use a photo from the DOTA val folder (the model never trained on it)
# that contains planes, ships and large vehicles.
#Step 9 already picks one like that.
# To get it onto your Mac, run this.

import os, shutil, config, tools
path, count = tools.find_demo_image(config.raw_dir, "val", config.my_class_ids)
out_dir = "test_image_for_streamlit"
os.makedirs(out_dir, exist_ok=True)
shutil.copy(path, out_dir)
print("saved:", os.path.join(out_dir, os.path.basename(path)), "| objects:", count)

saved: test_image_for_streamlit/P0179.jpg | objects: 86
